In [1]:
# ==========================================================
# DAY 18 - FINAL ANALYSIS & SUMMARY
# ==========================================================

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent

INPUT_FILE = PROJECT_ROOT / "03_processed_data/vendor_reliability.csv"
OUTPUT_FILE = PROJECT_ROOT / "03_processed_data/final_vendor_summary.csv"

df = pd.read_csv(INPUT_FILE)

print("Loaded:", df.shape)

# Rank vendors
df["rank"] = df["final_score"].rank(ascending=False)

# Top & Bottom vendors
top_vendors = df.sort_values("final_score", ascending=False).head(10)
low_vendors = df.sort_values("final_score").head(10)

# ----------------------------------------------------------
# DECISION FLAGS
# ----------------------------------------------------------

def decision(row):
    if row["risk_category"] == "High Risk":
        return "Replace Supplier"
    elif row["reliability_score"] > 0.7:
        return "Preferred Supplier"
    else:
        return "Monitor"

df["decision"] = df.apply(decision, axis=1)

# ----------------------------------------------------------
# KPI SUMMARY
# ----------------------------------------------------------

summary = {
    "Total Vendors": len(df),
    "High Risk Vendors": len(df[df["risk_category"] == "High Risk"]),
    "Preferred Vendors": len(df[df["decision"] == "Preferred Supplier"]),
    "Average Cost": df["avg_cost"].mean(),
    "Average Defect": df["avg_defect"].mean()
}

print("\nKPI SUMMARY:")
for k, v in summary.items():
    print(k, ":", v)

# Save KPI summary
pd.DataFrame([summary]).to_csv(
    PROJECT_ROOT / "03_processed_data/kpi_summary.csv",
    index=False
)

# ----------------------------------------------------------
# VISUAL 1 - Reliability Distribution
# ----------------------------------------------------------

plt.figure()

df["reliability_category"].value_counts().plot(kind="bar")

plt.title("Vendor Reliability Distribution")

plt.savefig(PROJECT_ROOT / "04_visualizations/reliability_distribution.png")
plt.close()

# ----------------------------------------------------------
# VISUAL 2 - Score Distribution
# ----------------------------------------------------------

plt.figure()

plt.hist(df["final_score"], bins=20)

plt.title("Final Score Distribution")

plt.savefig(PROJECT_ROOT / "04_visualizations/final_score_distribution.png")
plt.close()

# ----------------------------------------------------------
# PRINT OUTPUT
# ----------------------------------------------------------

print("\nTop Vendors:\n", top_vendors[["vendor_name", "final_score"]])
print("\nLow Performing Vendors:\n", low_vendors[["vendor_name", "final_score"]])

print("\nDay 18 Completed")

Loaded: (20, 19)

KPI SUMMARY:
Total Vendors : 20
High Risk Vendors : 2
Preferred Vendors : 4
Average Cost : 417.9157177999999
Average Defect : 6.2234336608

Top Vendors:
                            vendor_name  final_score
10       shimoga_agri_tech_cooperative     0.880264
3     belagavi_veg_growers_association     0.840801
7                     campco_mangaluru     0.818428
16  dakshina_kannada_exporters_network     0.774123
18                  tumkur_agro_supply     0.760833
13        chikkamagaluru_spice_traders     0.731563
2           bagalkot_onion_growers_fpc     0.730405
14            dharwad_seed_&_spice_fpo     0.648545
19     mandya_organic_farmers_alliance     0.647085
5                   malnad_nutri_foods     0.635032

Low Performing Vendors:
                           vendor_name  final_score
11        phondaghat_pharmacy_&_honey     0.271904
17                  kolar_farmers_fpo     0.282432
6                        raichur_apmc     0.334650
9   sahaja_samrudha_organi